# Rescued cNMF cluster reporter

This notebook is a fixed version of your original. Key changes:
- Robust handling of Perplexity API responses (status/JSON errors, missing key).
- Deep copy for payloads to avoid race conditions in threads.
- Adds a stable `annotation_granular` per record (falls back to the cluster name).
- Report builders no longer crash if fields are missing; they display clear errors instead.
- Optional: skip API calls if `PERPLEXITY_API_KEY` is not set; keeps everything reproducible.
- Convenience: save one file per report **and** a combined markdown; also export the queries to CSV.


In [1]:
import os, time, json, concurrent.futures
import requests
import pandas as pd
from copy import deepcopy

INPUT_CSV = './resources/cnmf_factor_cluster_top_genes_200.csv'
df = pd.read_csv(INPUT_CSV)
df.head()

,Cluster 0 genes,Cluster 0 # of shared factors,Cluster 1 genes,Cluster 1 # of shared factors,Cluster 2 genes,Cluster 2 # of shared factors,Cluster 3 genes,Cluster 3 # of shared factors,Cluster 4 genes,Cluster 4 # of shared factors,...,Cluster 30 genes,Cluster 30 # of shared factors,Cluster 31 genes,Cluster 31 # of shared factors,Cluster 32 genes,Cluster 32 # of shared factors,Cluster 33 genes,Cluster 33 # of shared factors,Cluster 34 genes,Cluster 34 # of shared factors
0,FAM189A2,16,TNC,14,DPP6,12,CFI,3,ALCAM,13,...,ABCA1,2,ABLIM3,1,WDR62,34,HSPH1,6,ATP13A4,5
1,OGFRL1,14,CD44,14,CHST11,11,SNTG1,3,GALNT13,12,...,FNDC3B,2,NHS,1,C21ORF58,34,PPP1R15A,6,LINC00299,5
2,MAP3K5,14,VCL,13,MAP3K1,11,PLEKHG1,3,DNM3,12,...,PKP4,2,MXD1,1,MSH5,34,UBC,6,AQP4,5
3,ITPR2,14,SAMD4A,13,SLC24A3,11,PRKD1,3,ADGRL3,12,...,PLCB1,2,MYOF,1,SMC4,34,CCDC59,6,HIF3A,5
4,ETNPPL,14,IGFBP7,13,NRXN1,11,ETV6,3,MIR181A2HG,12,...,FAM20C,2,NAMPT,1,LMNB1,34,UBB,6,LINC01727,5


In [2]:
# Build prompt objects from any column containing 'genes'
records = []
for c in df.columns:
    if 'genes' not in c.lower():
        continue
    gene_list = list(df[c].dropna())  # avoid NaNs
    o = {}
    o['cluster'] = c
    # Use the column header as our 'annotation_granular' fallback
    o['annotation_granular'] = c
    o['plain_query'] = (
        f"What might the following enriched gene list say about the type: {gene_list}"
    )
    o['contextual_query'] = (
        "The following gene list represents a gene program found in some subset of malignant cells isolated\n"
        "from an IDH-mutant astrocytoma. Please make predictions of how this gene program affects the malignant cells\n"
        "that express it — including structure, function (biological processes), metabolic state, interactions with the\n"
        "ECM and other cells. Use evidence not only from the astrocytoma literature and other relevant cancer literature,\n"
        "but also from the normal development and function of astrocytes.\n"
        "Rank predictions more highly where multiple genes in the list are known to be involved in a relevant process.\n"
        "Where multiple genes are known to be required for that process, assess whether all required genes are present and rank higher if they are.\n"
        + "Gene List: " + str(gene_list)
    )
    records.append(o)

len(records), records[0]['cluster'] if records else None

(35, 'Cluster 0 genes')

In [8]:
print(records[0])

{'cluster': 'Cluster 0 genes', 'annotation_granular': 'Cluster 0 genes', 'plain_query': "What might the following enriched gene list say about the type: ['FAM189A2', 'OGFRL1', 'MAP3K5', 'ITPR2', 'ETNPPL', 'NRG3', 'CD38', 'FMN2', 'LINC01088', 'KCNN3', 'DAAM2', 'AC002429.2', 'OBI1-AS1', 'NTRK2', 'SYTL4', 'WDR49', 'ADGRV1', 'LIFR', 'AQP4', 'ID3', 'OSBPL11', 'DPP10', 'SERPINI2', 'TLR4', 'NAA11', 'MGAT4C', 'AC026316.5', 'EEPD1', 'RASSF4', 'AL392086.3', 'SLC4A4', 'EDNRB', 'SLC39A11', 'ATP1A2', 'SLCO1C1', 'AHCYL2', 'SPON1', 'SLC1A3', 'GRAMD2B', 'DTNA', 'AC012405.1', 'NKAIN3', 'NTM', 'SLC14A1', 'DCLK2', 'DCLK1', 'ID4', 'AC124854.1', 'LINC01094', 'PCDH9', 'GABBR2', 'PARD3B', 'PDE8A', 'LRIG1', 'C5ORF64', 'RNF19A', 'SPARCL1', 'AC093535.1', 'FADS2', 'PLEKHA5', 'ASTN2', 'ADAMTS9', 'AC073941.1', 'SLC24A4', 'PAPPA', 'AC068587.4', 'FARP1', 'SORL1', 'ARHGAP26', 'CADPS', 'ST3GAL6', 'ITPKB', 'GABRB1', 'FAM107A', 'MIR99AHG', 'ANK2', 'AC107223.1', 'PPP2R2B', 'LPL', 'AL589935.1', 'MRVI1', 'TNIK', 'AL160272.

In [3]:
# --- API Setup ---
PPLX_KEY = os.getenv('PERPLEXITY_API_KEY')
PPLX_URL = 'https://api.perplexity.ai/chat/completions'

BASE_HEADERS = {
    'accept': 'application/json',
    'authorization': f'Bearer {PPLX_KEY}' if PPLX_KEY else '',
    'content-type': 'application/json',
}

BASE_PAYLOAD = {
    'model': 'sonar-deep-research',
    'return_citations': True,
    'search_domain_filter': [
        'pubmed.ncbi.nlm.nih.gov',
        'ncbi.nlm.nih.gov/pmc/',
        'sciencedirect.com',
        'nature.com',
        'cell.com',
        'frontiersin.org',
        'journals.plos.org',
        'wikipedia.org',
    ],
    'messages': [
        {'role': 'system', 'content': 'You are an expert biologist. Your answers must be based on primary scientific literature and major reviews from peer-reviewed sources.'},
        {'role': 'user', 'content': ''},
    ],
}

def safe_post(json_payload, timeout=60):
    try:
        resp = requests.post(PPLX_URL, headers=BASE_HEADERS, json=json_payload, timeout=timeout)
        if resp.status_code != 200:
            return {'error': f'HTTP {resp.status_code}', 'text': resp.text}
        try:
            return resp.json()
        except Exception as je:
            return {'error': f'Invalid JSON: {je}', 'text': resp.text[:1000]}
    except requests.exceptions.RequestException as re:
        return {'error': f'Request failed: {re}'}

def query_perplexity(o):
    # If no key, return clear error payloads so downstream code still works
    if not PPLX_KEY:
        o['plain_response'] = {'error': 'Missing PERPLEXITY_API_KEY'}
        o['contextual_response'] = {'error': 'Missing PERPLEXITY_API_KEY'}
        return o

    # Use a deep copy to avoid shared mutation between threads
    plx1 = deepcopy(BASE_PAYLOAD)
    plx1['messages'][1]['content'] = o['plain_query']
    plain_resp = safe_post(plx1)

    # small delay to be nicer to the API
    time.sleep(0.5)

    plx2 = deepcopy(BASE_PAYLOAD)
    plx2['messages'][1]['content'] = o['contextual_query']
    contextual_resp = safe_post(plx2)

    o['plain_response'] = plain_resp
    o['contextual_response'] = contextual_resp
    return o


In [4]:
print('Starting API calls...')
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
    for r in ex.map(query_perplexity, records):
        results.append(r)
print('All API calls completed.')
len(results)

Starting API calls...
All API calls completed.


35

In [5]:
def gen_bib(citations):
    if not citations:
        return '## References\n\nNo citations provided.'
    out = ['\n\n## References\n']
    for i, url in enumerate(citations, start=1):
        out.append(f'- [{i}] {url}')
    return '\n'.join(out)

def _title_from_row(row):
    return row.get('annotation_granular') or row.get('cluster') or 'Untitled cluster'

def rep(row, typ):
    """Creates one section of a report (plain or contextual)."""
    response_data = row.get(f'{typ}_response', {})
    title_txt = _title_from_row(row)
    # Error path
    if (not isinstance(response_data, dict)) or ('choices' not in response_data):
        error_message = response_data.get('error', 'Unknown error: response missing or malformed.') if isinstance(response_data, dict) else 'Non-dict response.'
        return f"## {typ.capitalize()} Query Report: {title_txt}\n\n**Error:**\n```\n{error_message}\n```"
    # Happy path
    content = response_data['choices'][0]['message'].get('content', 'No content found.')
    citations_list = response_data.get('citations', [])
    title = f"## {typ.capitalize()} Query Report: {title_txt}"
    query_text = f"**Query:**\n> {row.get(f'{typ}_query', '(missing)')}"
    bibliography = gen_bib(citations_list)
    return "\n\n".join([title, query_text, "**Response:**\n" + content, bibliography])

def generate_report(row):
    main_title = f"# Full Report for: {_title_from_row(row)}"
    return f"{main_title}\n\n---\n\n{rep(row, 'plain')}\n\n---\n\n{rep(row, 'contextual')}"

def generate_and_save_all_reports(records, directory_path):
    os.makedirs(directory_path, exist_ok=True)
    for r in records:
        base = _title_from_row(r).replace(' ', '_')
        plain_path = os.path.join(directory_path, f"{base}_plain.md")
        ctx_path = os.path.join(directory_path, f"{base}_contextual.md")
        with open(plain_path, 'w', encoding='utf-8') as f:
            f.write(rep(r, 'plain'))
        with open(ctx_path, 'w', encoding='utf-8') as f:
            f.write(rep(r, 'contextual'))
    # combined file
    combined = os.path.join(directory_path, 'ALL_REPORTS.md')
    with open(combined, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(generate_report(r))
            f.write('\n\n\n')
    return directory_path


In [6]:
print('--- Report preview (first record) ---')
if results:
    print(generate_report(results[0])[:1200] + '\n...')
else:
    print('No records built.')

--- Report preview (first record) ---
# Full Report for: Cluster 0 genes

---

## Plain Query Report: Cluster 0 genes

**Error:**
```
HTTP 401
```

---

## Contextual Query Report: Cluster 0 genes

**Error:**
```
HTTP 401
```
...


In [7]:
OUTPUT_DIR = './output/deepsearch_caroline/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
_ = generate_and_save_all_reports(results, OUTPUT_DIR)
print('Saved per-record and combined reports to:', OUTPUT_DIR)

# Also write queries to CSV for inspection/retries
pd.DataFrame(records)[['cluster','annotation_granular','plain_query','contextual_query']].to_csv(
    os.path.join(OUTPUT_DIR, 'queries.csv'), index=False
)
print('Saved queries.csv')

Saved per-record and combined reports to: ./output/deepsearch_caroline/
Saved queries.csv
